# 17

### LP - Problem

In [4]:
from docplex.mp.model import Model

mdl = Model(name="flow_lp")

# ---- helper to create variable dictionary ----
nodes = ["ab", "ad", "bc", "bd", "db", "dc", "de", "ed", "ea"]
periods = [1, 2, 3]

x = {(ij, k): mdl.continuous_var(lb=0, ub=1, name=f"x_{ij}^{k}")
     for ij in nodes for k in periods}

# ============= OBJECTIVE FUNCTION =============
obj = 0

# cost coefficients per period
costs = {
    1: {"ab":2, "ad":2, "bc":2, "bd":4, "db":2, "dc":16, "de":2, "ed":14, "ea":4},
    2: {"ab":2, "ad":2, "bc":2, "bd":4, "db":2, "dc":16, "de":2, "ed":14, "ea":4},
    3: {"ab":1, "ad":1, "bc":1, "bd":2, "db":1, "dc":8,  "de":1, "ed":7,  "ea":2},
}

for k in periods:
    for ij in nodes:
        obj += costs[k][ij] * x[(ij, k)]

mdl.minimize(obj)

# ============= CONSTRAINTS =============

# Period 1
mdl.add_constraint(x[("ea",1)] - x[("ab",1)] - x[("ad",1)] == 0)
mdl.add_constraint(x[("ab",1)] + x[("db",1)] - x[("bc",1)] - x[("bd",1)] == 0)
mdl.add_constraint(x[("bc",1)] + x[("dc",1)] == 0)

# Period 2
mdl.add_constraint(x[("ea",2)] - x[("ab",2)] - x[("ad",2)] == 0)
mdl.add_constraint(x[("ab",2)] + x[("db",2)] - x[("bc",2)] - x[("bd",2)] == 0)
mdl.add_constraint(x[("de",2)] - x[("ea",2)] - x[("ed",2)] == 0)

# Period 3
mdl.add_constraint(x[("ea",3)] - x[("ab",3)] - x[("ad",3)] == 0)
mdl.add_constraint(x[("ab",3)] + x[("db",3)] - x[("bc",3)] - x[("bd",3)] == 0)
mdl.add_constraint(
    x[("ad",3)] + x[("bd",3)] 
    - x[("ed",3)] - x[("de",3)]
    - x[("db",3)] - x[("dc",3)] == 0
)

# RHS = -1 constraints
mdl.add_constraint(x[("de",1)] - x[("ea",1)] - x[("ed",1)] == -1)
mdl.add_constraint(
    x[("ad",2)] + x[("bd",2)]
    - x[("ed",2)] - x[("de",2)]
    - x[("db",2)] - x[("dc",2)] == -1
)
mdl.add_constraint(x[("de",3)] - x[("ea",3)] - x[("ed",3)] == -1)

# RHS = +1 constraint
mdl.add_constraint(
    x[("ad",1)] + x[("bd",1)]
    - x[("ed",1)] - x[("de",1)]
    - x[("db",1)] - x[("dc",1)] == 1
)

# Flow = 1 constraints
mdl.add_constraint(x[("bc",2)] + x[("dc",2)] == 1)
mdl.add_constraint(x[("bc",3)] + x[("dc",3)] == 1)

# Capacity constraints
caps = {
    "ab": 4,
    "ad": 1,
    "bc": 5,
    "bd": 4,
    "db": 1,
    "dc": 3,
    "de": 3,
    "ed": 3,
    "ea": 5
}

for ij in nodes:
    mdl.add_constraint(2*x[(ij,1)] + 2*x[(ij,2)] + x[(ij,3)] <= caps[ij])

# ---- Solve ----
sol = mdl.solve(log_output=True)

if sol:
    print("Objective:", sol.objective_value)
    for ij in nodes:
        for k in periods:
            print(f"x_{ij}^{k} = {sol[x[(ij,k)]]}")
else:
    print("No solution. Status:", mdl.get_solve_status())


Version identifier: 22.1.2.0 | 2024-12-09 | 8bd2200c8
CPXPARAM_Read_DataCheck                          1
Tried aggregator 1 time.
LP Presolve eliminated 4 rows and 2 columns.
Aggregator did 5 substitutions.
Reduced LP has 15 rows, 20 columns, and 58 nonzeros.
Presolve time = 0.00 sec. (0.03 ticks)
Initializing dual steep norms . . .

Iteration log . . .
Iteration:     1   Dual objective     =            11.000000
Objective: 19.0
x_ab^1 = 0.5
x_ab^2 = 0.5
x_ab^3 = 1.0
x_ad^1 = 0.5
x_ad^2 = 0
x_ad^3 = 0
x_bc^1 = 0
x_bc^2 = 1.0
x_bc^3 = 1.0
x_bd^1 = 0.5
x_bd^2 = 0
x_bd^3 = 0
x_db^1 = 0
x_db^2 = 0.5
x_db^3 = 0
x_dc^1 = 0
x_dc^2 = 0
x_dc^3 = 0
x_de^1 = 0
x_de^2 = 0.5
x_de^3 = 0
x_ed^1 = 0
x_ed^2 = 0
x_ed^3 = 0
x_ea^1 = 1.0
x_ea^2 = 0.5
x_ea^3 = 1.0


# 18

### Integer programming problem

In [5]:
from docplex.mp.model import Model

mdl = Model(name="flow_mip")

# ---- helper to create binary variable dictionary ----
nodes = ["ab", "ad", "bc", "bd", "db", "dc", "de", "ed", "ea"]
periods = [1, 2, 3]

x = {(ij, k): mdl.binary_var(name=f"x_{ij}^{k}")
     for ij in nodes for k in periods}

# ============= OBJECTIVE FUNCTION =============
obj = 0

# cost coefficients
costs = {
    1: {"ab":2, "ad":2, "bc":2, "bd":4, "db":2, "dc":16, "de":2, "ed":14, "ea":4},
    2: {"ab":2, "ad":2, "bc":2, "bd":4, "db":2, "dc":16, "de":2, "ed":14, "ea":4},
    3: {"ab":1, "ad":1, "bc":1, "bd":2, "db":1, "dc":8,  "de":1, "ed":7,  "ea":2},
}

for k in periods:
    for ij in nodes:
        obj += costs[k][ij] * x[(ij, k)]

mdl.minimize(obj)

# ============= CONSTRAINTS =============

# -------- Period 1 --------
mdl.add_constraint(x[("ea",1)] - x[("ab",1)] - x[("ad",1)] == 0)
mdl.add_constraint(x[("ab",1)] + x[("db",1)] - x[("bc",1)] - x[("bd",1)] == 0)
mdl.add_constraint(x[("bc",1)] + x[("dc",1)] == 0)

# -------- Period 2 --------
mdl.add_constraint(x[("ea",2)] - x[("ab",2)] - x[("ad",2)] == 0)
mdl.add_constraint(x[("ab",2)] + x[("db",2)] - x[("bc",2)] - x[("bd",2)] == 0)
mdl.add_constraint(x[("de",2)] - x[("ea",2)] - x[("ed",2)] == 0)

# -------- Period 3 --------
mdl.add_constraint(x[("ea",3)] - x[("ab",3)] - x[("ad",3)] == 0)
mdl.add_constraint(x[("ab",3)] + x[("db",3)] - x[("bc",3)] - x[("bd",3)] == 0)
mdl.add_constraint(
    x[("ad",3)] + x[("bd",3)]
    - x[("ed",3)] - x[("de",3)]
    - x[("db",3)] - x[("dc",3)] == 0
)

# -------- RHS = -1 constraints --------
mdl.add_constraint(x[("de",1)] - x[("ea",1)] - x[("ed",1)] == -1)
mdl.add_constraint(
    x[("ad",2)] + x[("bd",2)]
    - x[("ed",2)] - x[("de",2)]
    - x[("db",2)] - x[("dc",2)] == -1
)
mdl.add_constraint(x[("de",3)] - x[("ea",3)] - x[("ed",3)] == -1)

# -------- RHS = +1 constraint --------
mdl.add_constraint(
    x[("ad",1)] + x[("bd",1)]
    - x[("ed",1)] - x[("de",1)]
    - x[("db",1)] - x[("dc",1)] == 1
)

# -------- Flow = 1 constraints --------
mdl.add_constraint(x[("bc",2)] + x[("dc",2)] == 1)
mdl.add_constraint(x[("bc",3)] + x[("dc",3)] == 1)

# -------- Capacity constraints --------
caps = {
    "ab": 4,
    "ad": 1,
    "bc": 5,
    "bd": 4,
    "db": 1,
    "dc": 3,
    "de": 3,
    "ed": 3,
    "ea": 5
}

for ij in nodes:
    mdl.add_constraint(2*x[(ij,1)] + 2*x[(ij,2)] + x[(ij,3)] <= caps[ij])

# ---- Solve ----
sol = mdl.solve(log_output=True)

if sol:
    print("Objective:", sol.objective_value)
    for ij in nodes:
        for k in periods:
            print(f"x_{ij}^{k} = {sol[x[(ij,k)]]}")
else:
    print("No solution. Status:", mdl.get_solve_status())


Version identifier: 22.1.2.0 | 2024-12-09 | 8bd2200c8
CPXPARAM_Read_DataCheck                          1
Tried aggregator 4 times.
MIP Presolve eliminated 13 rows and 13 columns.
MIP Presolve added 1 rows and 1 columns.
MIP Presolve modified 11 coefficients.
Aggregator did 6 substitutions.
Reduced MIP has 5 rows, 9 columns, and 19 nonzeros.
Reduced MIP has 8 binaries, 1 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.07 ticks)
Found incumbent of value 33.000000 after 0.00 sec. (0.08 ticks)
Probing fixed 3 vars, tightened 0 bounds.
Probing time = 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
MIP Presolve eliminated 4 rows and 5 columns.
MIP Presolve added 1 rows and 1 columns.
Reduced MIP has 2 rows, 5 columns, and 7 nonzeros.
Reduced MIP has 4 binaries, 1 generals, 0 SOSs, and 0 indicators.
Presolve time = 0.00 sec. (0.01 ticks)
Probing time = 0.00 sec. (0.00 ticks)
Tried aggregator 1 time.
MIP Presolve eliminated 1 rows and 1 columns.
MIP Presolve added 1 rows and 1

# 19

In [11]:
arcs = ["ab", "ad", "bc", "bd", "db", "dc", "de", "ed", "ea"]
arc_list = [(a[0], a[1]) for a in arcs]


In [14]:
from collections import defaultdict

# Build adjacency list
graph = defaultdict(list)
for u, v in arc_list:
    graph[u].append(v)

def all_simple_paths(graph, start, target):
    all_paths = []

    def dfs(current, path, visited):
        if current == target:
            all_paths.append(path.copy())
            return
        for nxt in graph[current]:
            if nxt not in visited:
                visited.add(nxt)
                path.append(nxt)
                dfs(nxt, path, visited)
                path.pop()
                visited.remove(nxt)

    dfs(start, [start], {start})
    return all_paths

paths_e_to_c = all_simple_paths(graph, "d", "c")

print("All paths from d to c:")
for p in paths_e_to_c:
    print(" -> ".join(p))


All paths from d to c:
d -> b -> c
d -> c
d -> e -> a -> b -> c


All simple paths from e to d:
e -> d
e -> a -> b -> d
e -> a -> d

All paths from d to c:
d -> b -> c
d -> c
d -> e -> a -> b -> c


e -> d -> b -> c
e -> d -> c
e -> a -> b -> c
e -> a -> b -> d -> c
e -> a -> d -> b -> c
e -> a -> d -> c

# 21

In [21]:
from docplex.mp.model import Model

# --- Model ---
mdl = Model(name="MINILINE_path_LP_weighted")

# Routes per commodity (matching your table)
R = {
    1: [1, 2],                 # k=1
    2: [1, 2],                 # k=2
    3: [1, 2, 3, 4, 5, 6]      # k=3
}

route_desc = {
    (1, 1): "e -> d",
    (1, 2): "e -> a -> b -> d",

    (2, 1): "d -> c",
    (2, 2): "d -> e -> a -> b -> c",

    (3, 1): "e -> d -> b -> c",
    (3, 2): "e -> d -> c",
    (3, 3): "e -> a -> b -> c",
    (3, 4): "e -> a -> b -> d -> c",
    (3, 5): "e -> a -> d -> b -> c",
    (3, 6): "e -> a -> d -> c",
}

# Demands / capacity usage per commodity
demand = {1: 2, 2: 2, 3: 1}

# --- Decision variables: LP relaxation, continuous in [0,1] ---
# x[k,r] = fraction of commodity k using route r
x = {(k, r): mdl.continuous_var(lb=0, ub=1, name=f"x_{k}_{r}")
     for k in R for r in R[k]}

# --- Path costs ---
cost = {
    (1, 1): 14,
    (1, 2): 10,

    (2, 1): 16,
    (2, 2): 10,

    (3, 1): 9,
    (3, 2): 15,
    (3, 3): 4,
    (3, 4): 13,
    (3, 5): 5,
    (3, 6): 11
}

# Objective: minimize total cost
mdl.minimize(mdl.sum(cost[k, r] * x[k, r] for k in R for r in R[k]))

# --- Capacity constraints with d1 = 2, d2 = 2, d3 = 1 ---

# (e,a): k1 r2; k2 r2; k3 r3,r4,r5,r6
mdl.add_constraint(
    2 * x[1, 2] +
    2 * x[2, 2] +
    (x[3, 3] + x[3, 4] + x[3, 5] + x[3, 6]) <= 5,
    "cap_ea"
)

# (a,b): k1 r2; k2 r2; k3 r3,r4
mdl.add_constraint(
    2 * x[1, 2] +
    2 * x[2, 2] +
    (x[3, 3] + x[3, 4]) <= 4,
    "cap_ab"
)

# (a,d): k3 r5,r6
mdl.add_constraint(
    x[3, 5] + x[3, 6] <= 1,
    "cap_ad"
)

# (b,c): k2 r2; k3 r1,r3,r5
mdl.add_constraint(
    2 * x[2, 2] +
    (x[3, 1] + x[3, 3] + x[3, 5]) <= 5,
    "cap_bc"
)

# (b,d): k1 r2; k3 r4
mdl.add_constraint(
    2 * x[1, 2] +
    x[3, 4] <= 4,
    "cap_bd"
)

# (d,b): k3 r1,r5
mdl.add_constraint(
    x[3, 1] + x[3, 5] <= 1,
    "cap_db"
)

# (d,c): k2 r1; k3 r2,r4,r6
mdl.add_constraint(
    2 * x[2, 1] +
    (x[3, 2] + x[3, 4] + x[3, 6]) <= 3,
    "cap_dc"
)

# (d,e): k2 r2
mdl.add_constraint(
    2 * x[2, 2] <= 3,
    "cap_de"
)

# (e,d): k1 r1; k3 r1,r2
mdl.add_constraint(
    2 * x[1, 1] +
    (x[3, 1] + x[3, 2]) <= 3,
    "cap_ed"
)

# --- Assignment: each commodity sends 1 unit of flow (can be split) ---

mdl.add_constraint(x[1, 1] + x[1, 2] == 1, "assign_k1")
mdl.add_constraint(x[2, 1] + x[2, 2] == 1, "assign_k2")
mdl.add_constraint(
    x[3, 1] + x[3, 2] + x[3, 3] + x[3, 4] + x[3, 5] + x[3, 6] == 1,
    "assign_k3"
)

# --- Solve LP ---
sol = mdl.solve(log_output=True)

if sol:
    print("Objective value (LP, path formulation):", sol.objective_value)
    print("\nNon-zero x_{k,r}:")
    for k in R:
        for r in R[k]:
            val = sol[x[k, r]]
            if abs(val) > 1e-6:
                print(f"x_{k}_{r} = {val}  ({route_desc[(k, r)]})")
else:
    print("No solution found. Status:", mdl.get_solve_status())


Version identifier: 22.1.2.0 | 2024-12-09 | 8bd2200c8
CPXPARAM_Read_DataCheck                          1
Tried aggregator 1 time.
LP Presolve eliminated 3 rows and 0 columns.
Aggregator did 2 substitutions.
Reduced LP has 7 rows, 8 columns, and 27 nonzeros.
Presolve time = 0.00 sec. (0.01 ticks)
Initializing dual steep norms . . .

Iteration log . . .
Iteration:     1   Dual objective     =            25.000000
Objective value (LP, path formulation): 25.0

Non-zero x_{k,r}:
x_1_2 = 1.0  (e -> a -> b -> d)
x_2_2 = 1.0  (d -> e -> a -> b -> c)
x_3_5 = 1.0  (e -> a -> d -> b -> c)


In [19]:
from docplex.mp.model import Model

# --- Model ---
mdl = Model(name="MINILINE_path_IP_weighted")

# Routes per commodity (matching your table)
# k = 1:
#   r1: e -> d
#   r2: e -> a -> b -> d
# k = 2:
#   r1: d -> c
#   r2: d -> e -> a -> b -> c
# k = 3:
#   r1: e -> d -> b -> c
#   r2: e -> d -> c
#   r3: e -> a -> b -> c
#   r4: e -> a -> b -> d -> c
#   r5: e -> a -> d -> b -> c
#   r6: e -> a -> d -> c

R = {
    1: [1, 2],
    2: [1, 2],
    3: [1, 2, 3, 4, 5, 6]
}

# For nice printing later
route_desc = {
    (1, 1): "e -> d",
    (1, 2): "e -> a -> b -> d",

    (2, 1): "d -> c",
    (2, 2): "d -> e -> a -> b -> c",

    (3, 1): "e -> d -> b -> c",
    (3, 2): "e -> d -> c",
    (3, 3): "e -> a -> b -> c",
    (3, 4): "e -> a -> b -> d -> c",
    (3, 5): "e -> a -> d -> b -> c",
    (3, 6): "e -> a -> d -> c",
}

# --- Demands / capacity usage per commodity (from original edge model) ---
# 2*x^1 + 2*x^2 + 1*x^3 in each capacity constraint
demand = {
    1: 2,   # k=1
    2: 2,   # k=2
    3: 1    # k=3
}

# --- Decision variables: binary ---
x = {(k, r): mdl.binary_var(name=f"x_{k}_{r}")
     for k in R for r in R[k]}

# --- Path costs \hat{c}^k_r (from arc costs) ---
cost = {
    (1, 1): 14,  # k=1, r1: e->d
    (1, 2): 10,  # k=1, r2: e->a->b->d

    (2, 1): 16,  # k=2, r1: d->c
    (2, 2): 10,  # k=2, r2: d->e->a->b->c

    (3, 1): 9,   # k=3, r1: e->d->b->c
    (3, 2): 15,  # k=3, r2: e->d->c
    (3, 3): 4,   # k=3, r3: e->a->b->c
    (3, 4): 13,  # k=3, r4: e->a->b->d->c
    (3, 5): 5,   # k=3, r5: e->a->d->b->c
    (3, 6): 11   # k=3, r6: e->a->d->c
}

# --- Objective: minimize total path cost ---
mdl.minimize(mdl.sum(cost[k, r] * x[k, r] for k in R for r in R[k]))

# --- Capacity constraints (with d1=2, d2=2, d3=1) ---
# Original capacities:
# 2 x_ab^1 + 2 x_ab^2 + x_ab^3 <= 4, etc.

# Arc (e,a): used by k1 r2; k2 r2; k3 r3,r4,r5,r6
# 2*x^1_ea + 2*x^2_ea + 1*x^3_ea <= 5
mdl.add_constraint(
    2 * x[1, 2]         # k1, r2
    + 2 * x[2, 2]       # k2, r2
    + 1 * (x[3, 3]      # k3, r3
           + x[3, 4]    # k3, r4
           + x[3, 5]    # k3, r5
           + x[3, 6])   # k3, r6
    <= 5,
    "cap_ea"
)

# Arc (a,b): used by k1 r2; k2 r2; k3 r3,r4
# 2*x_ab^1 + 2*x_ab^2 + x_ab^3 <= 4
mdl.add_constraint(
    2 * x[1, 2]         # k1, r2
    + 2 * x[2, 2]       # k2, r2
    + 1 * (x[3, 3]      # k3, r3
           + x[3, 4])   # k3, r4
    <= 4,
    "cap_ab"
)

# Arc (a,d): used by k3 r5,r6
# 2*x_ad^1 + 2*x_ad^2 + x_ad^3 <= 1 -> x_3,5 + x_3,6 <= 1
mdl.add_constraint(
    x[3, 5] + x[3, 6] <= 1,
    "cap_ad"
)

# Arc (b,c): used by k2 r2; k3 r1,r3,r5
# 2*x_bc^1 + 2*x_bc^2 + x_bc^3 <= 5
# Only k2 and k3 use bc here
mdl.add_constraint(
    2 * x[2, 2]         # k2, r2
    + 1 * (x[3, 1]      # k3, r1
           + x[3, 3]    # k3, r3
           + x[3, 5])   # k3, r5
    <= 5,
    "cap_bc"
)

# Arc (b,d): used by k1 r2; k3 r4
# 2*x_bd^1 + 2*x_bd^2 + x_bd^3 <= 4
# Only k1 and k3 use bd here
mdl.add_constraint(
    2 * x[1, 2]         # k1, r2
    + 1 * x[3, 4]       # k3, r4
    <= 4,
    "cap_bd"
)

# Arc (d,b): used by k3 r1,r5
# 2*x_db^1 + 2*x_db^2 + x_db^3 <= 1
# Only k3 uses db
mdl.add_constraint(
    x[3, 1] + x[3, 5] <= 1,
    "cap_db"
)

# Arc (d,c): used by k2 r1; k3 r2,r4,r6
# 2*x_dc^1 + 2*x_dc^2 + x_dc^3 <= 3
mdl.add_constraint(
    2 * x[2, 1]         # k2, r1
    + 1 * (x[3, 2]      # k3, r2
           + x[3, 4]    # k3, r4
           + x[3, 6])   # k3, r6
    <= 3,
    "cap_dc"
)

# Arc (d,e): used by k2 r2
# 2*x_de^1 + 2*x_de^2 + x_de^3 <= 3
# Only k2 uses de
mdl.add_constraint(
    2 * x[2, 2] <= 3,
    "cap_de"
)

# Arc (e,d): used by k1 r1; k3 r1,r2
# 2*x_ed^1 + 2*x_ed^2 + x_ed^3 <= 3
# Here, k1 and k3 use ed
mdl.add_constraint(
    2 * x[1, 1]         # k1, r1
    + 1 * (x[3, 1]      # k3, r1
           + x[3, 2])   # k3, r2
    <= 3,
    "cap_ed"
)

# --- Assignment: each commodity chooses exactly one route ---

# k = 1
mdl.add_constraint(x[1, 1] + x[1, 2] == 1, "assign_k1")

# k = 2
mdl.add_constraint(x[2, 1] + x[2, 2] == 1, "assign_k2")

# k = 3
mdl.add_constraint(
    x[3, 1] + x[3, 2] + x[3, 3] + x[3, 4] + x[3, 5] + x[3, 6] == 1,
    "assign_k3"
)

# --- Solve IP ---
sol = mdl.solve(log_output=True)

if sol:
    print("Objective value (IP, path formulation):", sol.objective_value)
    print("\nSelected routes:")
    for k in R:
        for r in R[k]:
            if sol[x[k, r]] > 0.5:  # binary
                print(f"Commodity {k}: route {r} ({route_desc[(k, r)]})")
else:
    print("No solution found. Status:", mdl.get_solve_status())


Version identifier: 22.1.2.0 | 2024-12-09 | 8bd2200c8
CPXPARAM_Read_DataCheck                          1
Found incumbent of value 39.000000 after 0.00 sec. (0.00 ticks)
Tried aggregator 2 times.
MIP Presolve eliminated 9 rows and 7 columns.
MIP Presolve modified 5 coefficients.
Aggregator did 3 substitutions.
All rows and columns eliminated.
Presolve time = 0.00 sec. (0.02 ticks)

Root node processing (before b&c):
  Real time             =    0.00 sec. (0.02 ticks)
Parallel b&c, 32 threads:
  Real time             =    0.00 sec. (0.00 ticks)
  Sync time (average)   =    0.00 sec.
  Wait time (average)   =    0.00 sec.
                          ------------
Total (root+branch&cut) =    0.00 sec. (0.02 ticks)
Objective value (IP, path formulation): 25.0

Selected routes:
Commodity 1: route 2 (e -> a -> b -> d)
Commodity 2: route 2 (d -> e -> a -> b -> c)
Commodity 3: route 5 (e -> a -> d -> b -> c)
